# 02 - K-Nearest Neighbors

**Owner:** Member B

**Inputs:** preprocessed splits produced by `00_eda_and_preprocessing.ipynb`
(`artifacts/splits/splits.npz`). Do not re-split or re-scale here.

**Outputs:** `artifacts/results/knn.json`,
`artifacts/models/knn.joblib`,
plus ROC / PR / confusion-matrix PNGs in `figures/`.

**Why this model**

KNN is a **non-parametric distance-based baseline**. It has no training stage — predictions are made by looking up the k closest training points by Euclidean / Manhattan distance. KNN benefits heavily from feature scaling (already applied in notebook 00), and is especially sensitive to class imbalance because dense minority regions get swamped by majority neighbours. KNN does not support `class_weight`, so Variant A here uses **distance-weighted neighbour voting** as a built-in soft rebalancing knob; Variant B applies SMOTE before KNN.

The structure below is identical across the five model notebooks:
1. Load shared splits.
2. **Variant A** — model with built-in imbalance handling (`class_weight`, `scale_pos_weight`, or distance weighting).
3. **Variant B** — SMOTE oversampling on training folds only (via `imblearn.pipeline.Pipeline`).
4. Pick the variant with the higher CV F1, refit, evaluate on the held-out test set.
5. Save the result JSON + the fitted model.


## 1. Setup & load shared splits

In [ ]:
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocess import load_splits
from src.tuning import grid_search, random_search, CV
from src.evaluation import (
    evaluate, save_results, print_metric_table,
    plot_confusion, plot_roc, plot_pr, _scores,
)

SPLITS_DIR  = PROJECT_ROOT / "artifacts" / "splits"
MODELS_DIR  = PROJECT_ROOT / "artifacts" / "models"
RESULTS_DIR = PROJECT_ROOT / "artifacts" / "results"
FIG_DIR     = PROJECT_ROOT / "figures"
for d in (MODELS_DIR, RESULTS_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

data = load_splits(SPLITS_DIR)
X_train, X_test = data["X_train"], data["X_test"]
y_train, y_test = data["y_train"], data["y_test"]
feature_names   = data["feature_names"]

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"train fraud rate: {y_train.mean():.4f}, test fraud rate: {y_test.mean():.4f}")


## 2. ### Variant A: no resampling, baseline KNN

Grid ranges over `n_neighbors`, `weights` (uniform vs inverse-distance), and `metric` (Euclidean vs Manhattan). Distance weighting is the closest substitute for `class_weight='balanced'`.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

estimator_cw = KNeighborsClassifier()  # KNN has no random state and no class_weight

grid_cw = {
    "n_neighbors": [3, 5, 7, 11, 21],
    "weights":     ["uniform", "distance"],
    "metric":      ["euclidean", "manhattan"],
}

search_cw = grid_search(estimator_cw, grid_cw)
search_cw.fit(X_train, y_train)
print(f"Variant A best CV F1: {search_cw.best_score_:.4f}  params: {search_cw.best_params_}")


## 3. ### Variant B: SMOTE + KNN

SMOTE inflates the minority class before KNN looks at neighbours, fighting the swamping problem head-on.

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

pipe_smote = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("clf",   KNeighborsClassifier()),
])

grid_smote = {
    "clf__n_neighbors": [3, 5, 7, 11, 21],
    "clf__weights":     ["uniform", "distance"],
    "clf__metric":      ["euclidean", "manhattan"],
}

search_smote = grid_search(pipe_smote, grid_smote)
search_smote.fit(X_train, y_train)
print(f"Variant B best CV F1: {search_smote.best_score_:.4f}  params: {search_smote.best_params_}")


## 4. Pick the better variant, refit & evaluate on test

In [ ]:
# Pick the variant with the better CV F1 score
variants = {
    "class_weight": search_cw,
    "smote":        search_smote,
}
best_variant = max(variants, key=lambda k: variants[k].best_score_)
best_search  = variants[best_variant]
best_model   = best_search.best_estimator_

print(f"\nWinning variant: {best_variant}")
print(f"  CV F1 (winning):       {best_search.best_score_:.4f}")
print(f"  CV F1 (other variant): {variants['smote' if best_variant=='class_weight' else 'class_weight'].best_score_:.4f}")
print(f"  Best params:           {best_search.best_params_}")

# Final evaluation on the held-out test set
results = evaluate(
    best_model,
    X_train, y_train, X_test, y_test,
    model_name="knn",
    best_params=best_search.best_params_,
    imbalance_strategy=best_variant,
)
print_metric_table(results)

save_results(results, RESULTS_DIR / "knn.json")
joblib.dump(best_model, MODELS_DIR / "knn.joblib")
print(f"\nSaved -> artifacts/results/knn.json + artifacts/models/knn.joblib")


## 5. Diagnostic plots

In [ ]:
# Diagnostic plots
y_pred = best_model.predict(X_test)
y_score = _scores(best_model, X_test)

fig_cm = plot_confusion(y_test, y_pred, title=f"{results['model_name']} - confusion matrix")
fig_cm.savefig(FIG_DIR / "knn_confusion.png", bbox_inches="tight")
plt.show()

if y_score is not None:
    fig_roc = plot_roc(y_test, y_score, label="knn")
    fig_roc.savefig(FIG_DIR / "knn_roc.png", bbox_inches="tight")
    plt.show()

    fig_pr = plot_pr(y_test, y_score, label="knn")
    fig_pr.savefig(FIG_DIR / "knn_pr.png", bbox_inches="tight")
    plt.show()


## 6. Hand-off to the comparison notebook

`artifacts/results/knn.json` now contains:

```
model_name, imbalance_strategy, best_params,
train: {accuracy, balanced_accuracy, precision, recall, f1, roc_auc, pr_auc},
test:  {accuracy, balanced_accuracy, precision, recall, f1, roc_auc, pr_auc},
confusion_matrix_test
```

`06_comparison_and_interpretation.ipynb` will read this file (alongside the
other four) to build the comparison table and overlay the ROC / PR curves.